In [ ]:
"""
Exercise 12-1: Genetic Algorithm with Parallel Fitness Evaluation
=================================================================
A simple Genetic Algorithm (GA) for function optimization, demonstrating
parallel evaluation of the fitness function across the population.

Optimization problem: Rastrigin function (2D)
    f(x, y) = 20 + x² + y² - 10·(cos(2πx) + cos(2πy))
Global minimum: f(0, 0) = 0, with many local minima.

Bayes' theorem connection:
    In the unified view of the lecture, evolution can be interpreted as
    implicit Bayesian inference:

        Selection  →  Likelihood    p(D|θ):  favor fitter individuals
        Mutation   →  Prior         p(θ):     explore new regions
        New pop    →  Posterior     p(θ|D):   next generation

    The fitness function f(x) plays the role of the likelihood:
        higher fitness = higher "probability" of being selected
        Selection probability ∝ f(x)  (analogous to p(D|θ))

    Parallelization: Each individual's fitness evaluation is independent,
    so pool.map() parallelizes the likelihood computation step.

Usage:
    python exercise_12_1_genetic_algorithm.py
"""

import time
import multiprocessing
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from numpy.random import default_rng

In [ ]:
# ===================================================================
# Objective function
# ===================================================================

def rastrigin(x: np.ndarray) -> float:
    """
    Rastrigin function (2D).

    Many local minima make this a challenging optimization problem.
    Global minimum at (0, 0) with f = 0.

    This function is the "likelihood" in our evolutionary interpretation:
    higher values are better, and each evaluation is independent.
    """
    return (
        20.0
        + x[0] ** 2 + x[1] ** 2
        - 10.0 * (np.cos(2 * np.pi * x[0]) + np.cos(2 * np.pi * x[1]))
    )

In [ ]:
# ===================================================================
# Genetic Algorithm
# ===================================================================

class GeneticAlgorithm:
    """
    Simple Genetic Algorithm with parallel fitness evaluation.

    Components:
      - Tournament selection
      - Blend crossover (alpha=0.5)
      - Gaussian mutation
      - Elitism (keep best individual)
    """

    def __init__(self, pop_size: int, bounds: list, mutation_rate: float = 0.1,
                 mutation_std: float = 0.5, crossover_rate: float = 0.8,
                 tournament_size: int = 3, elitism_count: int = 1,
                 num_workers: int = 1, seed: int = 42):
        self.pop_size = pop_size
        self.bounds = bounds
        self.mutation_rate = mutation_rate
        self.mutation_std = mutation_std
        self.crossover_rate = crossover_rate
        self.tournament_size = tournament_size
        self.elitism_count = elitism_count
        self.num_workers = num_workers
        self.seed = seed
        self.rng = default_rng(seed)

        # History tracking
        self.best_history = []
        self.mean_history = []
        self.best_individual_history = []

    def initialize_population(self) -> np.ndarray:
        """Random initialization within bounds."""
        lo, hi = self.bounds[0], self.bounds[1]
        return self.rng.uniform(lo, hi, (self.pop_size, 2))

    def evaluate_fitness_parallel(self, population: np.ndarray) -> np.ndarray:
        """
        Evaluate fitness of the entire population in parallel.

        This is the key parallelization point: each individual's fitness
        is computed independently using pool.map().

        In the Bayesian interpretation, this parallelizes the likelihood
        computation p(D|θ) for each candidate θ.
        """
        with multiprocessing.Pool(processes=self.num_workers) as pool:
            fitnesses = pool.map(rastrigin, population)
        return np.array(fitnesses)

    def tournament_select(self, population: np.ndarray,
                          fitnesses: np.ndarray) -> np.ndarray:
        """Tournament selection: pick the best of k random individuals."""
        indices = self.rng.choice(len(population), size=(len(population), self.tournament_size))
        tournament_fitnesses = fitnesses[indices]
        winners = indices[np.arange(len(population)), np.argmin(tournament_fitnesses, axis=1)]
        return population[winners]

    def blend_crossover(self, parent1: np.ndarray, parent2: np.ndarray) -> tuple:
        """Blend crossover (BLDE): offspring are convex combinations of parents."""
        if self.rng.random() > self.crossover_rate:
            return parent1.copy(), parent2.copy()
        alpha = self.rng.random()
        child1 = alpha * parent1 + (1 - alpha) * parent2
        child2 = alpha * parent2 + (1 - alpha) * parent1
        return child1, child2

    def mutate(self, individual: np.ndarray) -> np.ndarray:
        """Gaussian mutation with bounds clipping."""
        mutated = individual.copy()
        for dim in range(len(individual)):
            if self.rng.random() < self.mutation_rate:
                mutated[dim] += self.rng.normal(0, self.mutation_std)
                # Clip to bounds
                mutated[dim] = np.clip(mutated[dim], self.bounds[0], self.bounds[1])
        return mutated

    def evolve(self, num_generations: int) -> dict:
        """
        Run the genetic algorithm for num_generations.

        Each generation:
          1. Evaluate fitness (PARALLEL via pool.map)
          2. Select parents (tournament)
          3. Produce offspring (crossover + mutation)
          4. Elitism (carry forward best)
        """
        population = self.initialize_population()

        for gen in range(num_generations):
            # Step 1: Evaluate fitness (parallel)
            fitnesses = self.evaluate_fitness_parallel(population)

            # Track best
            best_idx = np.argmin(fitnesses)
            best_fit = fitnesses[best_idx]
            best_ind = population[best_idx].copy()
            mean_fit = np.mean(fitnesses)

            self.best_history.append(best_fit)
            self.mean_history.append(mean_fit)
            self.best_individual_history.append(best_ind.copy())

            # Step 2: Select parents
            parents = self.tournament_select(population, fitnesses)

            # Step 3: Produce offspring
            offspring = []
            i = 0
            while len(offspring) < self.pop_size:
                p1 = parents[i % len(parents)]
                p2 = parents[(i + 1) % len(parents)]
                c1, c2 = self.blend_crossover(p1, p2)
                offspring.append(self.mutate(c1))
                if len(offspring) < self.pop_size:
                    offspring.append(self.mutate(c2))
                i += 2

            population = np.array(offspring)

            # Step 4: Elitism
            if self.elitism_count > 0:
                # Find current best and replace worst offspring
                current_fitnesses = self.evaluate_fitness_parallel(population)
                worst_idx = np.argmax(current_fitnesses)
                population[worst_idx] = best_ind

        # Final evaluation
        fitnesses = self.evaluate_fitness_parallel(population)
        best_idx = np.argmin(fitnesses)

        return {
            "best_history": self.best_history,
            "mean_history": self.mean_history,
            "best_individual_history": self.best_individual_history,
            "best_individual": population[best_idx],
            "best_fitness": fitnesses[best_idx],
            "population": population,
        }

In [ ]:
# ===================================================================
# Visualization
# ===================================================================

def plot_contour_with_population(best_history: list, final_population: np.ndarray,
                                 bounds: list, output_dir: str = ".") -> str:
    """
    Contour plot of Rastrigin function with population overlay.
    Individuals colored by their generation (simulated by fitness rank).
    """
    x_range = np.linspace(bounds[0], bounds[1], 200)
    y_range = np.linspace(bounds[0], bounds[1], 200)
    X, Y = np.meshgrid(x_range, y_range)
    Z = np.array([rastrigin(np.array([x, y])) for x, y in zip(X.ravel(), Y.ravel())]).reshape(X.shape)

    fig, ax = plt.subplots(figsize=(10, 8))
    contour = ax.contourf(X, Y, Z, levels=50, cmap="viridis")
    plt.colorbar(contour, label="f(x, y)", ax=ax)

    # Plot final population
    colors = plt.cm.plasma(np.linspace(0, 1, len(final_population)))
    ax.scatter(final_population[:, 0], final_population[:, 1],
               c=colors, s=30, edgecolors="white", linewidths=0.5, alpha=0.8, label="Population")

    # Mark global minimum
    ax.scatter([0], [0], c="red", s=100, marker="*", label="Global min (0,0)", zorder=5)

    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("y", fontsize=12)
    ax.set_title("Genetic Algorithm: Population on Rastrigin Function", fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    path = f"{output_dir}/ga_contour.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_convergence(best_history: list, mean_history: list, num_workers: int,
                     output_dir: str = ".") -> str:
    """Plot best and mean fitness over generations."""
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(best_history, "b-", linewidth=2, label=f"Best fitness (workers={num_workers})")
    ax.plot(mean_history, "r--", linewidth=1.5, alpha=0.7, label="Mean fitness")
    ax.set_xlabel("Generation", fontsize=12)
    ax.set_ylabel("Fitness (Rastrigin)", fontsize=12)
    ax.set_title("Genetic Algorithm: Convergence", fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_yscale("log")

    path = f"{output_dir}/ga_convergence.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_speedup(results: list, output_dir: str = ".") -> str:
    """Plot execution time vs. number of workers."""
    workers = [r["workers"] for r in results]
    times = [r["time"] for r in results]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(workers, times, "bo-", markersize=8, linewidth=2)
    ax.set_xlabel("Number of Workers", fontsize=12)
    ax.set_ylabel("Execution Time (s)", fontsize=12)
    ax.set_title("GA: Parallel Speedup (Fitness Evaluation)", fontsize=14)
    ax.grid(True, alpha=0.3)

    for w, t in zip(workers, times):
        ax.annotate(f"{t:.3f}s", (w, t), textcoords="offset points",
                    xytext=(0, 12), ha="center", fontsize=9)

    path = f"{output_dir}/ga_speedup.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path

In [ ]:
# ===================================================================
# Main
# ===================================================================

def main():
    print("=" * 60)
    print("Exercise 12-1: Genetic Algorithm with multiprocessing")
    print("=" * 60)
    print()

    # --- Parameters ---
    pop_size = 100
    num_generations = 80
    bounds = [-5.12, 5.12]
    mutation_rate = 0.15
    mutation_std = 0.3
    num_workers_list = [1, 2, 4, 8]

    print(f"Optimization problem: Rastrigin function (2D)")
    print(f"  f(x,y) = 20 + x² + y² - 10·(cos(2πx) + cos(2πy))")
    print(f"  Global minimum: f(0,0) = 0")
    print()
    print(f"GA parameters: pop={pop_size}, generations={num_generations}, "
          f"mutation_rate={mutation_rate}, mutation_std={mutation_std}")
    print()

    # --- Bayes' theorem connection ---
    print("[1] Bayes' theorem in the Genetic Algorithm")
    print("    Evolutionary interpretation of Bayes' theorem:")
    print()
    print("      Selection  →  Likelihood    p(D|θ):  favor fitter individuals")
    print("      Mutation   →  Prior         p(θ):     explore new regions")
    print("      New pop    →  Posterior     p(θ|D):   next generation")
    print()
    print("    The fitness function acts as the likelihood:")
    print("      P(select individual i) ∝ f(x_i)")
    print("    This is analogous to Bayes' theorem where the likelihood")
    print("    p(D|θ) determines how strongly data supports each hypothesis.")
    print()

    # --- Main run ---
    print("[2] Running GA (workers=4) ...")
    ga = GeneticAlgorithm(
        pop_size=pop_size, bounds=bounds,
        mutation_rate=mutation_rate, mutation_std=mutation_std,
        num_workers=4, seed=42
    )
    start = time.perf_counter()
    result = ga.evolve(num_generations)
    elapsed = time.perf_counter() - start
    print(f"  Best fitness = {result['best_fitness']:.6f}")
    print(f"  Best solution = ({result['best_individual'][0]:.6f}, {result['best_individual'][1]:.6f})")
    print(f"  Time: {elapsed:.3f}s")
    print()

    # --- Visualization ---
    print("[3] Generating contour plot ...")
    contour_path = plot_contour_with_population(
        result["best_history"], result["population"], bounds
    )
    print(f"  Saved: {contour_path}")

    print("[4] Generating convergence plot ...")
    conv_path = plot_convergence(result["best_history"], result["mean_history"], 4)
    print(f"  Saved: {conv_path}")
    print()

    # --- Speedup benchmark ---
    print("[5] Parallel speedup benchmark")
    speedup_results = []
    for w in num_workers_list:
        ga_w = GeneticAlgorithm(
            pop_size=pop_size, bounds=bounds,
            mutation_rate=mutation_rate, mutation_std=mutation_std,
            num_workers=w, seed=42
        )
        start = time.perf_counter()
        res = ga_w.evolve(num_generations)
        elapsed = time.perf_counter() - start
        speedup_results.append({"workers": w, "time": elapsed,
                                "best_fitness": res["best_fitness"]})
        print(f"  workers={w:2d}: time={elapsed:.3f}s, best_fitness={res['best_fitness']:.6f}")

    print()
    print("[6] Generating speedup plot ...")
    speedup_path = plot_speedup(speedup_results)
    print(f"  Saved: {speedup_path}")
    print()

    # --- Summary ---
    print("[7] Summary: Bayes' theorem parallelization")
    print("    In GA, the fitness evaluation (analogous to likelihood p(D|θ))")
    print("    is computed independently for each individual. pool.map()")
    print("    parallelizes this computation, just as it parallelizes")
    print("    likelihood computations in the particle filter (Ex 11-2)")
    print("    and BO (Ex 12-2).")
    print()
    print("    Key insight: Bayes' theorem requires computing p(D|θ) for")
    print("    many candidate θ values — exactly what pool.map() does.")
    print()
    print("Done.")

In [ ]:
main()

Exercise 12-1: Genetic Algorithm with multiprocessing

Optimization problem: Rastrigin function (2D)
  f(x,y) = 20 + x² + y² - 10·(cos(2πx) + cos(2πy))
  Global minimum: f(0,0) = 0

GA parameters: pop=100, generations=80, mutation_rate=0.15, mutation_std=0.3

[1] Bayes' theorem in the Genetic Algorithm
    Evolutionary interpretation of Bayes' theorem:

      Selection  →  Likelihood    p(D|θ):  favor fitter individuals
      Mutation   →  Prior         p(θ):     explore new regions
      New pop    →  Posterior     p(θ|D):   next generation

    The fitness function acts as the likelihood:
      P(select individual i) ∝ f(x_i)
    This is analogous to Bayes' theorem where the likelihood
    p(D|θ) determines how strongly data supports each hypothesis.

[2] Running GA (workers=4) ...
  Best fitness = 0.000000
  Best solution = (0.000002, -0.000000)
  Time: 8.245s

[3] Generating contour plot ...
  Saved: ./ga_contour.png
[4] Generating convergence plot ...
  Saved: ./ga_convergence.png